In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset
import tiktoken
import json
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import re
from groq import Groq
import os
from dotenv import load_dotenv
import random

load_dotenv()  # Load environment variables from a .env file if present
client = Groq(api_key=os.getenv("groq_api_key"))

d:\Work\Github\google-tunix-kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
math =pd.read_parquet("../data/math-distillation/math_distillation_dataset.parquet")
code =pd.read_parquet("../data/code-distillation/code_distillation_dataset.parquet")
science =pd.read_parquet("../data/science/science_distillation_dataset.parquet")
common_sense =pd.read_parquet("../data/common-sense/commonsense_distillation_dataset.parquet")
creative_writing =pd.read_parquet("../data/creative-writing/creative_writing_distillation_dataset.parquet")
ideation =pd.read_parquet("../data/ideation/ideation_distillation_dataset.parquet")
financial = pd.read_parquet("../data/financial/financial_distillation_dataset.parquet")
reading = pd.read_parquet("../data/reading/reading_distillation_dataset.parquet")
summarization = pd.read_parquet("../data/summarization/summarization_distillation_dataset.parquet")
roleplay = pd.read_parquet("../data/roleplay/roleplay_distillation_dataset.parquet")
mtask = pd.read_parquet("../data/multitaskknowledge/multitaskknowledge_distillation_dataset.parquet")
convo = pd.read_parquet("../data/conversationalQA/conversationalQA_distillation_dataset.parquet")
#Function that adds a column called split if it's not there and sets it to 'train'
def ensure_split_column(df):
    if 'split' not in df.columns:
        df['split'] = 'train'
    return df
math = ensure_split_column(math)
code = ensure_split_column(code)
science = ensure_split_column(science)
common_sense = ensure_split_column(common_sense)
creative_writing = ensure_split_column(creative_writing)
ideation = ensure_split_column(ideation)
financial = ensure_split_column(financial)
reading = ensure_split_column(reading)
summarization = ensure_split_column(summarization)
roleplay = ensure_split_column(roleplay)
mtask = ensure_split_column(mtask)
convo = ensure_split_column(convo)


combined_df = pd.concat([math, code, science, common_sense, creative_writing, ideation, financial, reading, summarization
                         ,convo,mtask,roleplay], ignore_index=True)


In [22]:
##Function to clean noise in reasoning that was created either by the system prompt or indirectly created by model response

def clean_reasoning(text):
    # Handle None/NaN values
    if pd.isna(text) or text is None:
        return text

    # Clean up extra whitespace
    text = re.sub(r'\n\n+', '\n\n', text)
    text = re.sub(r'\s+\.', '.', text)  # Fix spacing before periods
    text = re.sub(r'\s+,', ',', text)  # Fix spacing before commas
    text = text.strip()
    
    # Only remove explicit "developer instructions" references
    #text = re.sub(r'developer instructions[:\s]*', '', text, flags=re.IGNORECASE)
    # Remove everything up to and including "developer instructions"
    text = re.sub(r'^.*?developer instructions[:\s]*', '', text, flags=re.IGNORECASE | re.MULTILINE)
    text= re.sub(r'^.*?"creative, specific, actionable ideas"[:\s]*', '', text, flags=re.IGNORECASE | re.MULTILINE)
    
    
    
    # Remove word count mentions - capture entire phrases
    text = re.sub(r'under \d+-\d+ words?\.?', '', text, flags=re.IGNORECASE)  # "under 200-300 words"
    text = re.sub(r'Keep (response |your response )?(under |within )?\d+-?\d* words?\.?', '', text, flags=re.IGNORECASE)  # "Keep 300-400 words", "Keep under 500 words"
    text = re.sub(r'\b\d+-\d+ words?\b', '', text)  # standalone "100-300 words"
    text = re.sub(r'~\d+ words?', '', text)          # "~180 words"
    text = re.sub(r'Word count target[^\n.]*\.?', '', text)
    text = re.sub(r'Should be under \d+\.?', '', text)
    text = re.sub(r'between \d+ and \d+ words?', '', text, flags=re.IGNORECASE)
    
    
    

    
    return text


##Adding a domain prefix to reasoning
##This creates a natural curriculum where the model learns distinct reasoning styles per domain.
##The domain prefix acts as a routing signal - essentially teaching the student model to self-identify the problem type and activate the appropriate reasoning mode.
def add_domain_prefix(reasoning, domain):
    """
    Add domain identification prefix to reasoning.
    
    Args:
        reasoning: The reasoning text
        domain: The domain (e.g., 'creative_ideation', 'summarization')
        
    Returns:
        str: Reasoning with domain prefix
    """
    if pd.isna(reasoning) or reasoning is None:
        return reasoning
    
    # Domain-specific prefixes
    domain_prefixes = {
        'code': 'This is a coding task.',
        'commonsense_reasoning': 'This is a commonsense reasoning task.',
        'creative_ideation': 'This is a creative ideation task.',
        'creative_writing': 'This is a creative writing task.',
        'financial_reasoning': 'This is a financial reasoning task.',
        'math': 'This is a mathematical reasoning task.',
        'numerical_reasoning': 'This is a numerical reasoning task.',
        'reading_comprehension': 'This is a reading comprehension task.',
        'science': 'This is a scientific reasoning task.',
        'summarization': 'This is a summarization task.'
    }
    
    # Get appropriate prefix
    prefix = domain_prefixes.get(domain, f'This is a {domain.replace("_", " ")} task.')
    
    # Add prefix to reasoning
    prefixed_reasoning = f"{prefix}\n\n{reasoning}"
    
    return prefixed_reasoning




##Filtering function to make sure we keep the content with quality reasoning
def is_low_quality_reasoning(reasoning, domain):
    # if domain != 'creative_ideation':
    #     return False
    
    if pd.isna(reasoning) or reasoning is None:
        return True
    
    # Remove ONLY meta-instruction phrases, not substantive content
    content = reasoning
    
    # Remove specific meta phrases (more precise patterns)
    meta_phrases = [
        r'We need to (provide|produce|create|predict|propose|brainstorm|respond|answer|generate)\b',
        r'Must be creative, specific, actionable\.?',
        r'Should be (within|under)\b[^.]*\.',
        r'Under \d+-\d+ words\.?',
        r'Keep (concise|within word limit)\.?',
        r'Provide (specific )?actionable ideas\.?',
        r'Also consider multiple perspectives\.?',
        r'Let\'s (craft|draft|produce)( about)?\.?\s*$',  # Only if it ends the text
        r'As a creative thinking expert,?',
    ]
    
    for phrase in meta_phrases:
        content = re.sub(phrase, '', content, flags=re.IGNORECASE)
    
    # Clean whitespace
    content = re.sub(r'\s+', ' ', content).strip()
    
    # Use character count (more reliable than word count)
    char_count = len(content)
    
    # Domain-specific thresholds
    thresholds = {
        'math': 80,  # Math reasoning is concise (calculations, formulas)
        'code': 80,  # Code reasoning can be brief (algorithm names, complexity)
        'numerical_reasoning': 80,
        'reading_comprehension': 100,
        'commonsense_reasoning': 100,
        'science': 100,
        'financial_reasoning': 150,
        'creative_ideation': 200,  # Creative needs more elaboration
        'creative_writing': 150,
        'summarization': 150,
    }
    
    threshold = thresholds.get(domain, 100)  # Default 100 chars
    
    if char_count < threshold:
        return True
    
    return False

In [23]:
##Clean the noise in reasoning
combined_df['cleaned_reasoning'] = combined_df['reasoning'].apply(clean_reasoning)

# Apply the function correctly
combined_df['low_reasoning_quality'] = combined_df.apply(
    lambda row: is_low_quality_reasoning(row['cleaned_reasoning'], row['domain']), 
    axis=1
)
# Add domain reasoning in
combined_df['cleaned_reasoning'] = combined_df.apply(
    lambda row: add_domain_prefix(row['cleaned_reasoning'], row['domain']),
    axis=1
)



In [15]:
combined_df.groupby(['domain','low_reasoning_quality']).size()

domain                 low_reasoning_quality
 MultiTask Knowledge   False                     4892
                       True                        17
code                   False                     7933
                       True                        41
commonsense_reasoning  False                     7208
                       True                        82
conversational         False                     2260
creative_ideation      False                     1819
                       True                       181
creative_writing       False                     3844
                       True                         2
financial_reasoning    False                     1674
                       True                       100
math                   False                    13104
                       True                      1718
numerical_reasoning    False                     2000
reading_comprehension  False                     3447
                       True          

In [ ]:
##Filter to train data samples and those with reasoning and response populated
combined_df1 = combined_df[(combined_df['split'] == 'train') & 
                          (combined_df['cleaned_reasoning'].notnull()) & 
                          (combined_df['response'].notnull())]

print(f"Filtered dataset size: {len(combined_df1)}")

#Filter to data with total_tokens < 1800
combined_df2 = combined_df1[combined_df1['total_tokens'] < 1800]

##Filter data with low reasoning content out
combined_df2 = combined_df2[combined_df2['low_reasoning_quality'] ==False]
print(f"Filtered dataset size after token limit: {len(combined_df2)}")

print(f"Total combined dataset size: {len(combined_df2)}")
print("Dataset sizes by domain:")
for domain in combined_df2['domain'].unique():
    domain_size = len(combined_df2[combined_df2['domain'] == domain])
    print(f"  {domain}: {domain_size}")
    
print("Dataset sizes by source:")
for source in combined_df2['source'].unique():
    source_size = len(combined_df2[combined_df2['source'] == source])
    print(f"  {source}: {source_size}")

Filtered dataset size: 58824
Filtered dataset size after token limit: 52967
Total combined dataset size: 52967
Dataset sizes by domain:
  math: 10385
  code: 6348
  science: 4367
  commonsense_reasoning: 6372
  creative_writing: 3825
  creative_ideation: 1797
  numerical_reasoning: 1997
  financial_reasoning: 1665
  reading_comprehension: 3398
  summarization: 1871
  conversational: 2259
   MultiTask Knowledge: 4832
  roleplay: 3851
Dataset sizes by source:
  gsm8k: 7423
  OpenR1-Math-220k: 2317
  Math/AIME 2025: 5
  ChilleD/SVAMP: 640
  google-research-datasets/mbpp: 324
  nvidia/OpenCodeReasoning: 100
  codeparrot/apps: 107
  sahil2801/CodeAlpaca-20k: 2945
  iamtarun/python_code_instructions_18k_alpaca: 2872
  ai2_arc/ARC-Challenge: 1115
  sciq: 2958
  Idavidrein/gpqa: 294
  tau/commonsense_qa: 1999
  tasksource/strategy-qa: 2281
  hotpot_qa: 2092
  euclaise/writingprompts: 1830
  igormorgado/ROCStories2018: 1995
  synthetic_ideation_prompts: 1797
  ibm/tatqa: 1997
  FinGPT/fingpt-co

##Train - Validation - Test Split

In [25]:
from sklearn.model_selection import train_test_split

# Samples filtered out because split != 'train'
non_train_split = combined_df[(combined_df['split'] != 'train') & 
                               (combined_df['reasoning'].notnull()) & 
                               (combined_df['response'].notnull())]
print(f"Samples from non-train splits: {len(non_train_split)}")

# Samples that passed initial filters but exceeded token limit
dropped_samples = combined_df1[combined_df1['total_tokens'] >= 1900]
print(f"Samples dropped due to token limit: {len(dropped_samples)}")

# Your filtered training candidates
combined_df2 = combined_df1[combined_df1['total_tokens'] < 1900]
print(f"Samples for train/valid/test split: {len(combined_df2)}")

# First split: 90% train, 10% temp (which will become 5% valid, 5% test)
train_df, temp_df = train_test_split(
    combined_df2, 
    train_size=0.95, 
    stratify=combined_df2['domain'],
    random_state=42
)

# Second split: split the 10% temp into 50-50 (5% valid, 5% test of original)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.7,
    stratify=temp_df['domain'],
    random_state=42
)

# Add dropped samples and non-train splits to test set
test_df = pd.concat([test_df, dropped_samples, non_train_split], ignore_index=True)

print(f"\nFinal split:")
print(f"Train: {len(train_df)} ({len(train_df)/len(combined_df2)*100:.1f}%)")
print(f"Valid: {len(valid_df)} ({len(valid_df)/len(combined_df2)*100:.1f}%)")
print(f"Test: {len(test_df)} samples (base: {len(temp_df)//2}, +{len(dropped_samples)} long-context, +{len(non_train_split)} non-train splits)")

Samples from non-train splits: 600
Samples dropped due to token limit: 4493
Samples for train/valid/test split: 54331

Final split:
Train: 51614 (95.0%)
Valid: 815 (1.5%)
Test: 6995 samples (base: 1358, +4493 long-context, +600 non-train splits)


In [26]:
print(f"Total combined dataset size: {len(train_df)}")
print("Dataset sizes by domain:")
for domain in train_df['domain'].unique():
    domain_size = len(train_df[train_df['domain'] == domain])
    print(f"  {domain}: {domain_size}")

Total combined dataset size: 51614
Dataset sizes by domain:
  conversational: 2147
  math: 10118
  creative_writing: 3637
  science: 4202
  financial_reasoning: 1682
  roleplay: 3677
  reading_comprehension: 3296
   MultiTask Knowledge: 4615
  commonsense_reasoning: 6356
  numerical_reasoning: 1898
  code: 6151
  summarization: 1942
  creative_ideation: 1893


In [27]:
train_df.to_parquet('../data//final_data//custom_df_train_sft.parquet')
valid_df.to_parquet('../data//final_data//custom_df_valid_sft.parquet')
test_df.to_parquet('../data//final_data//custom_df_test_sft.parquet')

### Run the quality assesment on the dataset

In [30]:
train_df_quality_checked = pd.read_parquet('../data//final_data//custom_df_train_sft_quality_tested.parquet')

    
print(f"\nFiltering from {len(train_df_quality_checked)} samples...")
##Selecting top quality samples only with atleast 7 scores
    
# Quality filters
filtered = train_df_quality_checked[
    (train_df_quality_checked['correctness'] >= 7) &
    (train_df_quality_checked['reasoning_quality'] >= 7) &
    (train_df_quality_checked['difficulty'] != 'UNKNOWN')
].copy()
    
    
    
# Statistics
print(f"\nFinal Dataset:")
print(f"  Samples: {len(filtered)}")
print(f"  Avg Correctness: {filtered['correctness'].mean():.2f}/10")
print(f"  Avg Reasoning: {filtered['reasoning_quality'].mean():.2f}/10")

difficulty_dist = filtered['difficulty'].value_counts()
print(f"\nDifficulty:")
for diff, count in difficulty_dist.items():
    print(f"  {diff}: {count} ({count/len(filtered)*100:.1f}%)")
    



Filtering from 51614 samples...

Final Dataset:
  Samples: 50609
  Avg Correctness: 9.53/10
  Avg Reasoning: 8.52/10

Difficulty:
  MEDIUM: 36179 (71.5%)
  EASY: 14313 (28.3%)
  HARD: 117 (0.2%)


In [62]:
# ##Add any other column we need to the training filtered data from tarin_df
# train_df2= pd.merge(filtered,train_df[['input','problem_type','question_type','source']],on=['input'],how='inner')

##Add the unaswerable questions also in traing and we are good
unaswerable = pd.read_parquet('../data/unanswerable/unknown_distillation_dataset.parquet')
unaswerable['cleaned_reasoning']=np.where(unaswerable['expected_behavior']=="Say I don't know",unaswerable['reasoning']+'. We must not hallucinate',unaswerable['reasoning'])
['correctness', 'reasoning_quality', 'difficulty']
unaswerable['correctness']=10
unaswerable['reasoning_quality']=10
unaswerable['difficulty']='HARD'
unaswerable['domain']=np.where(unaswerable['domain']=="knowledge"," MultiTask Knowledge",unaswerable['domain'])
train_df2= pd.concat([filtered,unaswerable[filtered.columns]],ignore_index=True)

In [61]:
unaswerable.groupby('domain').size()

domain
commonsense_reasoning     25
knowledge                 25
reading_comprehension    120
science                   30
dtype: int64

In [63]:
train_df2.groupby('domain').size()

domain
 MultiTask Knowledge     4557
code                     6030
commonsense_reasoning    6236
conversational           2109
creative_ideation        1858
creative_writing         3569
financial_reasoning      1638
math                     9917
numerical_reasoning      1872
reading_comprehension    3352
roleplay                 3615
science                  4151
summarization            1905
dtype: int64

In [65]:
train_df2.to_parquet("../data//final_data//custom_df_train_sft_quality_filtered.parquet")

Evalidation Sample

In [8]:
valid_df = pd.read_parquet("../data//final_data//custom_df_valid_sft.parquet")
# Define your target counts per domain
domain_counts = {
    'code': 10,
    'math': 10,
    'commonsense_reasoning': 20,
    'creative_ideation': 20,
    'creative_writing': 20,
    'financial_reasoning': 10,
    'numerical_reasoning': 10,
    'reading_comprehension': 20,
    'science': 20,
    'summarization': 20
}

# Sample each domain separately and concatenate
sampled_dfs = []
for domain, count in domain_counts.items():
    domain_samples = valid_df[valid_df['domain'] == domain].sample(
        n=min(count, len(valid_df[valid_df['domain'] == domain])),
        random_state=42  # for reproducibility
    )
    sampled_dfs.append(domain_samples)

valid_df_eval_sample = pd.concat(sampled_dfs, ignore_index=True)
valid_df_eval_sample.groupby('domain').size()

domain
code                     10
commonsense_reasoning    20
creative_ideation        20
creative_writing         20
financial_reasoning      10
math                     10
numerical_reasoning      10
reading_comprehension    20
science                  20
summarization            20
dtype: int64

In [11]:
valid_df_eval_sample=valid_df_eval_sample[['uid','input','ground_truth','domain','problem_type','reasoning','response']]
valid_df_eval_sample.to_parquet("../data//final_data//model_evaluation_sample.parquet")